# 7주차 텍스트 데이터 시각화
합성 댓글 120건과 합성 뉴스 기사 60건 — 실제 수집 자료가 아님.

## [공통-1] 라이브러리와 한글 글꼴 설치

In [ ]:
!pip -q install wordcloud pandas matplotlib requests
!apt-get -qq install fonts-nanum > /dev/null

## [공통-2] 라이브러리 불러오기

In [ ]:
import io, re, requests
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from IPython.display import display
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
assert Path(font_path).exists(), '먼저 설치 셀을 실행하세요'
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
Path('results').mkdir(exist_ok=True)

## [공통-3] 댓글·뉴스 CSV 읽기. 실패할 경우 파일 업로드

In [ ]:
BASE = 'https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week07/02_%EB%8D%B0%EC%9D%B4%ED%84%B0/'
def load_csv(filename):
    try:
        res = requests.get(BASE + filename, timeout=30)
        res.raise_for_status()
        return pd.read_csv(io.BytesIO(res.content), encoding='utf-8-sig')
    except Exception as e:
        print('CSV 다운로드 실패:', e)
        from google.colab import files
        uploaded = files.upload()
        return pd.read_csv(io.BytesIO(uploaded[filename]), encoding='utf-8-sig')
comments=load_csv('youtube_comments_sample.csv')
articles=load_csv('news_articles_sample.csv')
print('댓글:',len(comments),'뉴스:',len(articles))
display(comments.head(3),articles.head(3))

## [공통-4] 단순 정규식 토큰화와 단어 빈도 함수. 형태소 분석과 다릅니다.

In [ ]:
STOPWORDS={'그리고','하지만','정말','조금','위한','있습니다','합니다','대한','함께','위해','했다','있다','것'}
def tokens(text):
    words=re.findall(r'[가-힣A-Za-z]{2,}',str(text).lower())
    return [w for w in words if w not in STOPWORDS]
def analyze(series,stem):
    freq=Counter(w for text in series.fillna('').astype(str) for w in tokens(text))
    top=pd.DataFrame(freq.most_common(15),columns=['word','frequency'])
    display(top)
    top.to_csv(f'results/{stem}_top15.csv',index=False,encoding='utf-8-sig')
    fig,ax=plt.subplots(figsize=(11,5))
    ax.bar(top['word'],top['frequency']);ax.tick_params(axis='x',rotation=45)
    fig.tight_layout();fig.savefig(f'results/{stem}_bars.png',dpi=150);plt.show()
    wc=WordCloud(font_path=font_path,width=1100,height=600,background_color='white',random_state=42).generate_from_frequencies(freq)
    plt.figure(figsize=(11,6));plt.imshow(wc);plt.axis('off');plt.show()
    wc.to_file(f'results/{stem}_wordcloud.png')
    return top,freq

## [실습1-1] 댓글 주제별 건수

In [ ]:
display(comments.groupby('topic').size().reset_index(name='count'))

## [실습1-2] 댓글 전처리

In [ ]:
comments['tokens']=comments['text'].apply(tokens)
display(comments[['text','tokens']].head())

## [실습1-3]~[실습1-4] 댓글 단어 빈도와 워드클라우드

In [ ]:
comment_top,comment_freq=analyze(comments['text'],'youtube_comments')

## [실습1-5] AI 비교를 위한 프롬프트 생성 — 출력된 문구를 ChatGPT에 복사

In [ ]:
def ai_prompt(series,label):
    sample='\n'.join(series.fillna('').astype(str).drop_duplicates().head(15).tolist())
    return f'다음 {label} 내용만 근거로 핵심을 세 문장으로 요약하고 의미 기반 키워드 다섯 개와 각각의 원문 근거를 제시하세요. 없는 사실은 추정하지 마세요.\n'+sample
print(ai_prompt(comments['text'],'유튜브 댓글'))

## [실습2-1] 뉴스 제목과 본문 합치기

In [ ]:
articles['full_text']=articles['title'].fillna('')+' '+articles['body'].fillna('')
display(articles[['title','full_text']].head(3))

## [실습2-2] 뉴스 카테고리별 건수

In [ ]:
display(articles.groupby('category').size().reset_index(name='count'))

## [실습2-3] 뉴스 워드클라우드

In [ ]:
news_top,news_freq=analyze(articles['full_text'],'news_articles')

## [실습2-4] AI 뉴스 요약 키워드 프롬프트

In [ ]:
print(ai_prompt(articles['full_text'],'뉴스 텍스트'))

## [실습3-1] 실제 AI 결과를 직접 입력하여 키워드 비교

In [ ]:
comment_ai_keywords=[]  # 위 AI 출력에서 키워드 5개 입력
news_ai_keywords=[]
def compare(top,keywords,label):
    frequency_set=set(top['word'].head(5))
    ai_set=set(w.lower() for w in keywords)
    print(label,'빈도:',frequency_set,'AI:',ai_set,'완전 일치:',frequency_set&ai_set)
compare(comment_top,comment_ai_keywords,'유튜브')
compare(news_top,news_ai_keywords,'뉴스')

## [실습3-2] 결과물 ZIP 다운로드

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('Week07_results','zip','results')
files.download('Week07_results.zip')

## [선택] AI API 자동 실행 — 비용 발생·키 보안 주의

In [ ]:
RUN_OPENAI_API=False
if RUN_OPENAI_API:
    !pip -q install openai
    from google.colab import userdata
    from openai import OpenAI
    client=OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
    response=client.responses.create(model='gpt-4.1-mini',input=ai_prompt(comments['text'],'댓글'))
    print(response.output_text)
else:
    print('API 자동 실행 생략: 앞에서 만든 프롬프트를 ChatGPT에 복사하세요.')